# 04 — TOC-Aware Chunking

Build metadata-rich chunks using the PDF TOC hierarchy. The default `legacy` strategy preserves
historical chunk IDs and boundaries; `targeted` is an explicit bounded alternative.

Part page spans describe the text actually present, including overlap. Legacy splitting preserves historical paragraph boundaries and can exceed the requested maximum for a single long paragraph. Targeted splitting bounds body characters before the Section prefix. PDFs need a usable built-in TOC and extractable text; OCR and no-TOC fallback are not supported.


In [ ]:
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "src" / "intelligent_scada").is_dir():
            return candidate
    raise RuntimeError("Could not locate the Intelligent SCADA project root.")

project_root = find_project_root(Path.cwd().resolve())
src_dir = project_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from intelligent_scada.config import Settings

settings = Settings.from_env(project_root=project_root, env_override=True).for_default_document()
settings.ensure_output_dirs()


In [ ]:
from intelligent_scada.artifacts import read_json
from intelligent_scada.chunking.toc_chunker import create_chunks

structure = read_json(settings.pdf_structure_path)
cleaned_pages = read_json(settings.clean_text_path)

chunks = create_chunks(
    structure,
    cleaned_pages,
    settings.text_chunks_path,
    target=settings.chunk_target_size,
    maximum=settings.chunk_max_size,
    overlap=settings.chunk_overlap,
    strategy=settings.chunking_strategy,
    skip_sections=settings.skip_sections,
)

if not chunks:
    raise ValueError("No chunks: verify usable TOC bookmarks and extractable text.")

sizes = [chunk["char_count"] for chunk in chunks]
print(f"Chunks: {len(chunks)}")
print(f"Average characters: {sum(sizes) / len(sizes):.0f}")
print(f"Range: {min(sizes)}-{max(sizes)}")
print(f"Output: {settings.text_chunks_path}")
